In [ ]:
# PUBLIC_MATERIALS_NOTEBOOK_SETUP
from pathlib import Path
from urllib.request import urlretrieve

try:
    import google.colab  # type: ignore  # noqa: F401
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

_MATERIALS_BASE = \"https://raw.githubusercontent.com/yonpak/ai-from-zero-to-builder-materials/main/labs/notebooks/tiny-llm/\"
if _IN_COLAB or not Path(\"tiny_llm.py\").exists():
    for _name in (\"tiny_llm.py\", \"tiny_corpus.txt\"):
        urlretrieve(_MATERIALS_BASE + _name, _name)


# Build a Simple Tokenizer

This notebook supports `l04-04-build-a-simple-tokenizer`.

It uses the original fixture `tiny_corpus.txt`. The tokenizer is deliberately simple:
lowercase text, split words/numbers/punctuation, build a sorted vocabulary, and reserve id `0` for `<unk>`.

In [ ]:
from pathlib import Path
from tiny_llm import SimpleTokenizer

text = Path("tiny_corpus.txt").read_text(encoding="utf-8")
tokenizer = SimpleTokenizer.from_text(text)

print("vocab size:", tokenizer.vocab_size)
print("first 15 vocab items:", tokenizer.itos[:15])

In [ ]:
sample = "The model predicts the next token."
ids = tokenizer.encode(sample)
tokens = tokenizer.decode_tokens(ids)

print(tokens)
print(ids)
print(tokenizer.decode(ids))

assert len(tokens) == len(ids)
assert tokenizer.encode("zzzxxyy") == [0]
assert tokenizer.decode(tokenizer.encode(sample)) == sample.lower()

## Break it on purpose

Try a word that never appeared in the corpus. It becomes `<unk>`.
That is not a bug in this tiny tokenizer; it is the limitation we want to notice before learning subword tokenization.

In [ ]:
unknown_ids = tokenizer.encode("the zzzxxyy token")
print(unknown_ids)
print(tokenizer.decode_tokens(unknown_ids))
assert 0 in unknown_ids

## Validation

Success means:

- the same corpus always creates the same sorted vocabulary;
- encode/decode is stable for in-vocabulary text after lowercasing and simple punctuation spacing;
- unseen tokens map to `<unk>` id `0`.

No GPU is used.